In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score


# 2. Load Breast Cancer Dataset

data = load_breast_cancer()

X = data.data
y = data.target

print("Dataset Shape:", X.shape)
print("Target Shape:", y.shape)


# 3. Create Train, Validation and Test Splits

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print("\nTraining Data:", X_train.shape)
print("Validation Data:", X_val.shape)
print("Testing Data:", X_test.shape)


# 4. Train Logistic Regression (Baseline)

baseline = LogisticRegression(max_iter=10000)

baseline.fit(X_train, y_train)

y_pred = baseline.predict(X_val)

baseline_accuracy = accuracy_score(y_val, y_pred)

print("\nBaseline Accuracy:", baseline_accuracy)


# 5. Perform 5-Fold Cross Validation

cv_scores = cross_val_score(
    baseline, X_train, y_train, cv=5, scoring='accuracy'
)

print("\nCross Validation Scores:", cv_scores)
print("Mean CV Accuracy:", cv_scores.mean())


# 6. Create Pipeline with StandardScaler and Logistic Regression

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=10000))
])

pipeline.fit(X_train, y_train)

pipeline_pred = pipeline.predict(X_val)

pipeline_accuracy = accuracy_score(y_val, pipeline_pred)

print("\nPipeline Accuracy:", pipeline_accuracy)


# 7. Hyperparameter Tuning using GridSearchCV

param_grid = {
    'classifier__C': [0.01, 0.1, 1, 10, 100]
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='accuracy'
)

grid_search.fit(X_train, y_train)

print("\nBest Parameters:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)


# 8. Evaluate Tuned Model

best_model = grid_search.best_estimator_

tuned_pred = best_model.predict(X_val)

tuned_accuracy = accuracy_score(y_val, tuned_pred)

print("\nTuned Validation Accuracy:", tuned_accuracy)


# 9. Final Testing

test_pred = best_model.predict(X_test)

test_accuracy = accuracy_score(y_test, test_pred)

print("\nFinal Test Accuracy:", test_accuracy)


# 10. Comparison Table

results = pd.DataFrame({
    'Model': [
        'Baseline Logistic Regression',
        '5-Fold Cross Validation',
        'StandardScaler Pipeline',
        'Tuned Logistic Regression'
    ],
    'Accuracy': [
        baseline_accuracy,
        cv_scores.mean(),
        pipeline_accuracy,
        tuned_accuracy
    ]
})

print("\nModel Comparison:")
print(results)

Dataset Shape: (569, 30)
Target Shape: (569,)

Training Data: (341, 30)
Validation Data: (114, 30)
Testing Data: (114, 30)

Baseline Accuracy: 0.9649122807017544

Cross Validation Scores: [0.92753623 0.94117647 0.92647059 0.98529412 0.91176471]
Mean CV Accuracy: 0.9384484228473999

Pipeline Accuracy: 0.9912280701754386

Best Parameters: {'classifier__C': 0.1}
Best CV Score: 0.9736146632566071

Tuned Validation Accuracy: 0.9824561403508771

Final Test Accuracy: 0.9824561403508771

Model Comparison:
                          Model  Accuracy
0  Baseline Logistic Regression  0.964912
1       5-Fold Cross Validation  0.938448
2       StandardScaler Pipeline  0.991228
3     Tuned Logistic Regression  0.982456
